In [ ]:

import inspect

candidate_names = [
    "extract_landmarks",
    "extract_features",
    "process_video",
    "process_video_landmarks",
    "extract_video_features",
    "get_hand_landmarks",
    "get_pose_landmarks"
]

for name in candidate_names:
    obj = globals().get(name)

    if callable(obj):
        print(f"\n{'='*50}")
        print("FOUND:", name)

        try:
            print(inspect.getsource(obj))
        except (OSError, TypeError):
            print("Function exists, but its source is unavailable.")

print("\nCheck complete.")


In [ ]:

import os
import glob
import json
import numpy as np

BATCH_DIR = "/content/drive/MyDrive/ASL_Citizen_Project/batches"
batch_files = sorted(glob.glob(os.path.join(BATCH_DIR, "batch_*.npz")))

print("Batch files:", len(batch_files))

if batch_files:
    with np.load(batch_files[0], allow_pickle=False) as data:
        print("\nFirst batch arrays:")
        for key in data.files:
            print(key, data[key].shape, data[key].dtype)

        print("\nFirst metadata record:")
        print(json.loads(data["metadata"][0]))

        features = data["features"]
        print("\nFirst video's feature shape:", features[0].shape)
        print("First frame, first 20 features:")
        print(features[0, 0, :20])
        print("First frame, hand features:")
        print(features[0, 0, :126])


In [ ]:

import numpy as np

sample = features[0]

hand = sample[:, :126]
pose = sample[:, 126:]

print("VIDEO:", "GUESS1")
print("Feature shape:", sample.shape)

print("\nFrame-wise feature detection")
for i in range(sample.shape[0]):
    hand_detected = np.any(hand[i] != 0)
    pose_detected = np.any(pose[i] != 0)

    print(
        f"Frame {i + 1}: "
        f"Hand detected = {hand_detected}, "
        f"Pose detected = {pose_detected}, "
        f"Hand nonzero values = {np.count_nonzero(hand[i])}, "
        f"Pose nonzero values = {np.count_nonzero(pose[i])}"
    )

print("\nOverall hand nonzero values:", np.count_nonzero(hand), "/", hand.size)
print("Overall pose nonzero values:", np.count_nonzero(pose), "/", pose.size)


In [ ]:

import numpy as np

hand_all = X[:, :, :126]
pose_all = X[:, :, 126:]

hand_frame_active = np.any(hand_all != 0, axis=2)
pose_frame_active = np.any(pose_all != 0, axis=2)

hand_frames_per_video = hand_frame_active.sum(axis=1)
pose_frames_per_video = pose_frame_active.sum(axis=1)

print("TOTAL VIDEOS:", len(X))
print("TOTAL FRAMES:", X.shape[0] * X.shape[1])

print("\nHAND FRAME COVERAGE")
print("Frames with hand features:", int(hand_frame_active.sum()))
print("Hand detection rate:", round(float(hand_frame_active.mean() * 100), 2), "%")
print("Videos with 0 detected frames:", int((hand_frames_per_video == 0).sum()))
print("Videos with 1-3 detected frames:", int(((hand_frames_per_video >= 1) & (hand_frames_per_video <= 3)).sum()))
print("Videos with 4-8 detected frames:", int((hand_frames_per_video >= 4).sum()))

print("\nPOSE FRAME COVERAGE")
print("Frames with pose features:", int(pose_frame_active.sum()))
print("Pose detection rate:", round(float(pose_frame_active.mean() * 100), 2), "%")
print("Videos with 0 detected frames:", int((pose_frames_per_video == 0).sum()))

print("\nHAND FRAMES PER VIDEO DISTRIBUTION")
for count in range(9):
    print(f"{count} frames:", int((hand_frames_per_video == count).sum()))


In [ ]:

import os
import glob

video_name = "24946258334528815-GUESS.mp4"

search_roots = [
    "/content/ASL_Citizen",
    "/content/drive/MyDrive/ASL_Citizen_Project",
    "/content/drive/MyDrive"
]

matches = []

for root in search_roots:
    if os.path.exists(root):
        matches.extend(
            glob.glob(os.path.join(root, "**", video_name), recursive=True)
        )

matches = list(dict.fromkeys(matches))

print("Matching original videos:", len(matches))

for path in matches[:10]:
    print(path)


In [ ]:

import os

search_roots = [
    "/content/drive/MyDrive",
    "/content/drive/Shareddrives"
]

target_names = {
    "train.csv",
    "val.csv",
    "test.csv",
    "ASL_Citizen.zip"
}

found = []

for root in search_roots:
    if not os.path.exists(root):
        continue

    for current_dir, dirs, files in os.walk(root):
        dirs[:] = [
            d for d in dirs
            if d not in {"node_modules", ".git", ".cache"}
        ]

        for filename in files:
            if filename in target_names:
                found.append(os.path.join(current_dir, filename))

print("Matching dataset files:", len(found))

for path in found[:50]:
    print(path)


In [ ]:

import os
import glob

paths_to_check = [
    "/content/asl_citizen_features/models",
    "/content/drive/MyDrive/ASL_Citizen_Project/models",
    "/content/drive/MyDrive/ASL_Citizen_Project/batches",
]

for folder in paths_to_check:
    print(f"\n{folder}")
    if not os.path.exists(folder):
        print("Folder not found")
        continue

    for path in sorted(glob.glob(os.path.join(folder, "*")))[:20]:
        if os.path.isfile(path):
            print(os.path.basename(path), "-", round(os.path.getsize(path) / (1024 * 1024), 2), "MB")
        else:
            print(os.path.basename(path), "- folder")

print("\nCurrent runtime model-related files:")
for pattern in [
    "/content/**/*hand*landmarker*",
    "/content/**/*pose*landmarker*",
    "/content/**/*.keras",
    "/content/**/*.tflite",
]:
    for path in glob.glob(pattern, recursive=True)[:10]:
        if os.path.isfile(path):
            print(path)


In [ ]:

import os
import numpy as np
import tensorflow as tf

model_path = "/content/asl_outputs/asl_baseline_model.keras"

print("MODEL EXISTS:", os.path.exists(model_path))

if os.path.exists(model_path):
    print("Model size (MB):", round(os.path.getsize(model_path) / (1024 * 1024), 2))
    saved_model = tf.keras.models.load_model(model_path)
    print("Model input shape:", saved_model.input_shape)
    print("Model output shape:", saved_model.output_shape)

print("\nFEATURE GROUP CHECK")
sample = X[0]

groups = {
    "Features 1-63": sample[:, :63],
    "Features 64-126": sample[:, 63:126],
    "Features 127-258": sample[:, 126:258],
}

for name, values in groups.items():
    active_frames = np.any(values != 0, axis=1)
    print(f"\n{name}")
    print("Shape:", values.shape)
    print("Active frames:", int(active_frames.sum()), "/", len(active_frames))
    print("Nonzero values:", int(np.count_nonzero(values)))
    print("Standard deviation:", round(float(np.std(values)), 5))


In [ ]:

import glob
import os
import numpy as np

batch_files = sorted(
    glob.glob(
        "/content/drive/MyDrive/ASL_Citizen_Project/batches/batch_*.npz"
    )
)

total_videos = 0
group1_active = 0
group2_active = 0
pose_active = 0
group2_nonzero = 0

for path in batch_files:
    with np.load(path, allow_pickle=False) as data:
        features = data["features"]

        group1 = features[:, :, :63]
        group2 = features[:, :, 63:126]
        pose = features[:, :, 126:]

        total_videos += len(features)
        group1_active += int(np.any(group1 != 0, axis=2).sum())
        group2_active += int(np.any(group2 != 0, axis=2).sum())
        pose_active += int(np.any(pose != 0, axis=2).sum())
        group2_nonzero += int(np.count_nonzero(group2))

print("Batches checked:", len(batch_files))
print("Videos checked:", total_videos)
print("Group 1 active frames:", group1_active)
print("Group 2 active frames:", group2_active)
print("Group 2 nonzero values:", group2_nonzero)
print("Pose active frames:", pose_active)


In [ ]:

import numpy as np

hand1 = X[:, :, :63]
hand2 = X[:, :, 63:126]

for name, group in [
    ("Group 1", hand1),
    ("Group 2", hand2)
]:
    active_frames = np.any(group != 0, axis=2)
    active_values = group[group != 0]

    print(f"\n{name}")
    print("Active frames:", int(active_frames.sum()))
    print("Nonzero values:", len(active_values))

    if len(active_values) > 0:
        print("Minimum:", round(float(active_values.min()), 5))
        print("Maximum:", round(float(active_values.max()), 5))
        print("Mean:", round(float(active_values.mean()), 5))
        print("Standard deviation:", round(float(active_values.std()), 5))


In [ ]:

import numpy as np

hand1_active = np.any(X[:, :, :63] != 0, axis=2)
hand2_active = np.any(X[:, :, 63:126] != 0, axis=2)

patterns = {
    "Group 1 only": hand1_active & ~hand2_active,
    "Group 2 only": ~hand1_active & hand2_active,
    "Both groups": hand1_active & hand2_active,
    "Neither group": ~hand1_active & ~hand2_active,
}

print("TOTAL FRAMES:", X.shape[0] * X.shape[1])

for name, mask in patterns.items():
    count = int(mask.sum())
    print(f"{name}: {count} ({count / mask.size * 100:.2f}%)")

print("\nGroup 2 active without Group 1:",
      bool(np.any(~hand1_active & hand2_active)))


In [ ]:

import numpy as np

group1 = X[:, :, :63]
group2 = X[:, :, 63:126]

def inspect_landmark_structure(name, group):
    # Reshape 63 values into 21 landmarks with 3 coordinates each
    landmarks = group.reshape(-1, 8, 21, 3)

    print(f"\n{name}")
    print("Reshaped:", landmarks.shape)

    for coordinate, index in [("Coordinate 1", 0),
                               ("Coordinate 2", 1),
                               ("Coordinate 3", 2)]:
        values = landmarks[:, :, :, index]
        nonzero = values[values != 0]

        print(f"{coordinate} nonzero:", len(nonzero))
        if len(nonzero):
            print("  min:", round(float(nonzero.min()), 5))
            print("  max:", round(float(nonzero.max()), 5))
            print("  mean:", round(float(nonzero.mean()), 5))

inspect_landmark_structure("Group 1", group1)
inspect_landmark_structure("Group 2", group2)


In [ ]:

import numpy as np

for name, group in [
    ("Group 1", X[:, :, :63]),
    ("Group 2", X[:, :, 63:126])
]:
    landmarks = group.reshape(-1, 8, 21, 3)

    # A landmark is considered empty if all 3 coordinates are zero
    empty_landmarks = np.all(landmarks == 0, axis=3)

    # A frame is empty if all 21 landmarks are zero
    empty_frames = np.all(empty_landmarks, axis=2)

    print(f"\n{name}")
    print("Total landmarks:", empty_landmarks.size)
    print("Empty landmarks:", int(empty_landmarks.sum()))
    print("Landmarks with coordinates:", int((~empty_landmarks).sum()))
    print("Fully empty frames:", int(empty_frames.sum()))
    print("Frames with at least one landmark:",
          int((~empty_frames).sum()))

    detected_per_frame = (~empty_landmarks).sum(axis=2)
    print("Mean landmarks per frame:",
          round(float(detected_per_frame.mean()), 2))


In [ ]:

import os
import json
import numpy as np
import pandas as pd

output_dir = "/content/drive/MyDrive/ASL_Citizen_Project"
os.makedirs(output_dir, exist_ok=True)

hand1_active = np.any(X[:, :, :63] != 0, axis=2)
hand2_active = np.any(X[:, :, 63:126] != 0, axis=2)
pose_active = np.any(X[:, :, 126:] != 0, axis=2)

report = {
    "feature_records": int(X.shape[0]),
    "frames_per_video": int(X.shape[1]),
    "features_per_frame": int(X.shape[2]),
    "hand_group_1_active_frames": int(hand1_active.sum()),
    "hand_group_2_active_frames": int(hand2_active.sum()),
    "pose_active_frames": int(pose_active.sum()),
    "hand_group_1_detection_rate": float(hand1_active.mean()),
    "hand_group_2_detection_rate": float(hand2_active.mean()),
    "pose_detection_rate": float(pose_active.mean()),
    "test_accuracy": 0.1312,
    "test_macro_f1": 0.1088,
    "notes": [
        "Recovered subset contains 3150 feature records.",
        "Original source videos and official split CSVs were not found in checked locations.",
        "Hand group identities have not been verified from the extraction code.",
        "Test metrics are baseline results, not state-of-the-art claims."
    ]
}

report_path = os.path.join(output_dir, "experiment_diagnostics.json")

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("Saved diagnostics:", report_path)
print("Records:", report["feature_records"])
print("Test accuracy:", report["test_accuracy"])
print("Test macro-F1:", report["test_macro_f1"])


In [ ]:

import json
import os

notebook_path = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "ASL_Signer_Independent_Multimodal_Model.ipynb"
)

terms = [
    "HandLandmarker",
    "PoseLandmarker",
    "hand_landmarker.task",
    "pose_landmarker.task",
    "detect_for_video",
    "detect_for_image",
    "hand_landmarks",
    "pose_landmarks",
    "extract_landmarks",
    "extract_features",
]

if not os.path.exists(notebook_path):
    print("Notebook not found:", notebook_path)
else:
    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    matches = 0

    for i, cell in enumerate(notebook.get("cells", [])):
        source = "".join(cell.get("source", []))

        found = [term for term in terms if term.lower() in source.lower()]

        if found:
            matches += 1
            print(f"\n--- CELL {i} | Matches: {', '.join(found)} ---")
            print(source[:5000])

    print("\nMatching cells:", matches)


In [ ]:

import os

roots = [
    "/content",
    "/content/drive/MyDrive/ASL_Citizen_Project"
]

extensions = (".task", ".py")

for root in roots:
    print(f"\nSearching: {root}")

    if not os.path.exists(root):
        print("Path not found")
        continue

    found = 0

    for current, dirs, files in os.walk(root):
        dirs[:] = [
            d for d in dirs
            if d not in {"node_modules", ".git", ".cache"}
        ]

        for filename in files:
            if filename.lower().endswith(extensions):
                path = os.path.join(current, filename)
                print(path)
                found += 1

    print("Matching files:", found)


In [ ]:

from pathlib import Path

file_path = Path("/content/WLASL/code/TGCN/gen_features.py")

if file_path.exists():
    print(file_path.read_text(encoding="utf-8")[:12000])
else:
    print("File not found:", file_path)


In [ ]:

import importlib.util
import os

print("MediaPipe installed:", importlib.util.find_spec("mediapipe") is not None)

if importlib.util.find_spec("mediapipe") is not None:
    import mediapipe as mp
    print("MediaPipe version:", mp.__version__)

print("\nPossible task files:")
for root in ["/content", "/content/drive/MyDrive"]:
    for current, dirs, files in os.walk(root):
        dirs[:] = [
            d for d in dirs
            if d not in {"node_modules", ".git", ".cache"}
        ]

        for filename in files:
            if filename in {"hand_landmarker.task", "pose_landmarker.task"}:
                print(os.path.join(current, filename))


In [ ]:

import os
import glob

project_dir = "/content/drive/MyDrive/ASL_Citizen_Project"

for folder in [
    project_dir,
    os.path.join(project_dir, "batches"),
    "/content/asl_outputs"
]:
    print(f"\n{folder}")

    if not os.path.exists(folder):
        print("Folder not found")
        continue

    for path in sorted(glob.glob(os.path.join(folder, "*"))):
        if os.path.isfile(path):
            print(
                os.path.basename(path),
                f"({os.path.getsize(path) / (1024 * 1024):.2f} MB)"
            )
        elif os.path.isdir(path):
            print(os.path.basename(path), "(folder)")


In [ ]:

import os

print("100-class landmark model in memory:",
      "model" in globals())

if "model" in globals():
    print("Model input shape:", model.input_shape)
    print("Model output shape:", model.output_shape)

print("\nSaved model files in project folders:")

for root in [
    "/content/drive/MyDrive/ASL_Citizen_Project/models",
    "/content/asl_outputs"
]:
    print("\n", root)

    if os.path.exists(root):
        files = [
            f for f in os.listdir(root)
            if f.endswith((".keras", ".h5", ".weights.h5"))
        ]

        if files:
            for filename in files:
                print(filename)
        else:
            print("No saved neural-network model files found.")
    else:
        print("Folder not found.")


In [ ]:

import os
import json

project_dir = "/content/drive/MyDrive/ASL_Citizen_Project"
os.makedirs(project_dir, exist_ok=True)

model_path = os.path.join(
    project_dir,
    "asl_100class_landmark_lstm.keras"
)

# Save the trained 100-class landmark model
model.save(model_path)

print("Model saved:", model_path)
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("File size:", round(os.path.getsize(model_path) / (1024 * 1024), 2), "MB")


In [ ]:

import json
import os

project_dir = "/content/drive/MyDrive/ASL_Citizen_Project"

if "label_encoder" in globals():
    class_names = label_encoder.classes_.tolist()
elif "metadata" in globals():
    from sklearn.preprocessing import LabelEncoder

    label_encoder = LabelEncoder()
    label_encoder.fit(metadata["Gloss"].astype(str))
    class_names = label_encoder.classes_.tolist()
else:
    raise NameError(
        "Neither label_encoder nor metadata is available. "
        "Do not create a new label mapping blindly."
    )

if len(class_names) != model.output_shape[-1]:
    raise ValueError(
        f"Mismatch: {len(class_names)} labels, "
        f"{model.output_shape[-1]} model outputs."
    )

labels_path = os.path.join(
    project_dir,
    "asl_100class_label_mapping.json"
)

with open(labels_path, "w", encoding="utf-8") as f:
    json.dump(
        {str(i): name for i, name in enumerate(class_names)},
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved label mapping:", labels_path)
print("Number of classes:", len(class_names))
print("First 10 labels:", class_names[:10])


In [ ]:

import os
import json
import numpy as np
import pandas as pd

project_dir = "/content/drive/MyDrive/ASL_Citizen_Project"
os.makedirs(project_dir, exist_ok=True)

results = {
    "experiment": "100-class ASL landmark LSTM",
    "input_shape": list(model.input_shape[1:]),
    "output_classes": int(model.output_shape[-1]),
    "recovered_feature_records": 3150,
    "expected_records": 3218,
    "missing_records": 68,
    "test_accuracy": 0.1312,
    "test_macro_f1": 0.1088,
    "best_validation_accuracy": 0.1616,
    "best_validation_loss": 3.0346,
    "notes": [
        "Signer-independent split",
        "Recovered subset; 68 expected records are missing",
        "Results are from the current baseline experiment"
    ]
}

results_path = os.path.join(
    project_dir, "experiment_results.json"
)

with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("Experiment results saved:", results_path)

if "history" in globals() and hasattr(history, "history"):
    history_df = pd.DataFrame(history.history)
    history_path = os.path.join(
        project_dir, "training_history.csv"
    )
    history_df.to_csv(history_path, index=False)
    print("Training history saved:", history_path)
else:
    print("Training history variable unavailable; not recreated.")

print("\nSaved model exists:",
      os.path.exists(os.path.join(
          project_dir, "asl_100class_landmark_lstm.keras"
      )))
print("Saved label mapping exists:",
      os.path.exists(os.path.join(
          project_dir, "asl_100class_label_mapping.json"
      )))


In [ ]:

required_variables = [
    "model",
    "X_test",
    "y_test",
    "label_encoder"
]

print("TEST EVALUATION READINESS\n")

for name in required_variables:
    available = name in globals()
    print(f"{name}: {'Available' if available else 'Missing'}")

if "X_test" in globals():
    print("X_test shape:", X_test.shape)

if "y_test" in globals():
    print("y_test shape:", y_test.shape)

if "label_encoder" in globals():
    print("Label encoder classes:", len(label_encoder.classes_))


In [ ]:

import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

project_dir = "/content/drive/MyDrive/ASL_Citizen_Project"
results_dir = os.path.join(project_dir, "evaluation")
os.makedirs(results_dir, exist_ok=True)

# Verify model and label compatibility
class_names = label_encoder.classes_

assert model.output_shape[-1] == len(class_names), \
    "Model output count and label count do not match."

assert X_test.shape[0] == len(y_test), \
    "Test features and labels have different lengths."

assert np.min(y_test) >= 0 and np.max(y_test) < len(class_names), \
    "Test labels are outside the valid class range."

# Generate predictions
probabilities = model.predict(X_test, verbose=1)
y_pred = np.argmax(probabilities, axis=1)

# Overall metrics
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(
    y_test, y_pred, average="macro", zero_division=0
)

print("\nOVERALL TEST RESULTS")
print("Test samples:", len(y_test))
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Macro-F1: {macro_f1:.4f}")

# Per-class report
report = classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(class_names)),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()
report_path = os.path.join(results_dir, "classification_report.csv")
report_df.to_csv(report_path)

# Confusion matrix
cm = confusion_matrix(
    y_test, y_pred, labels=np.arange(len(class_names))
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)
cm_path = os.path.join(results_dir, "confusion_matrix.csv")
cm_df.to_csv(cm_path, index_label="actual_gloss")

# Individual predictions
predictions_df = pd.DataFrame({
    "actual_class_id": y_test.astype(int),
    "actual_gloss": class_names[y_test.astype(int)],
    "predicted_class_id": y_pred.astype(int),
    "predicted_gloss": class_names[y_pred],
    "correct": y_test == y_pred,
    "confidence": np.max(probabilities, axis=1)
})

predictions_path = os.path.join(results_dir, "test_predictions.csv")
predictions_df.to_csv(predictions_path, index=False)

# Save a separate metrics summary
summary = {
    "test_samples": int(len(y_test)),
    "number_of_classes": int(len(class_names)),
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1)
}

import json

with open(
    os.path.join(results_dir, "evaluation_summary.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)

print("\nFILES SAVED")
print("Classification report:", report_path)
print("Confusion matrix:", cm_path)
print("Test predictions:", predictions_path)
print("Evaluation summary:",
      os.path.join(results_dir, "evaluation_summary.json"))

print("\nTOP 10 CLASSES BY F1-SCORE")
print(
    report_df.loc[class_names, ["precision", "recall", "f1-score", "support"]]
    .sort_values("f1-score", ascending=False)
    .head(10)
    .round(3)
)
